# Optimized Tech-Centric Resume Classification System (Gradio UI)
This notebook trains an optimized resume classifier on the **Resume Dataset** (snehaanbhawal/resume-dataset) from Kaggle. It implements text processing with custom stopwords to minimize category confusion, compares multiple classification models, prints the training curves and confusion matrix, and launches a Gradio UI testing playground at the end.

## Step 1: Install Dependencies and Download NLTK data

In [ ]:
# Install dependencies (including Gradio for the testing playground)
!pip install -q pandas numpy scikit-learn matplotlib seaborn joblib nltk kaggle gradio

import re
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression, SGDClassifier
from sklearn.svm import LinearSVC
from sklearn.naive_bayes import MultinomialNB
from sklearn.model_selection import train_test_split, cross_val_score, learning_curve
from sklearn.metrics import classification_report, confusion_matrix

# Download NLTK datasets for cleaning text
for package in ("stopwords", "wordnet", "omw-1.4"):
    nltk.download(package, quiet=True)

## Step 2: Download the Resume Dataset from Kaggle
This step downloads the **Resume Dataset** (`snehaanbhawal/resume-dataset`) from Kaggle. Provide your Kaggle API credentials (`kaggle.json`) when prompted to authorize the download.

In [ ]:
import os
import json

print("--- Kaggle Setup ---")
print("Option 1: Upload your 'kaggle.json' file.")
print("Option 2: Cancel/Skip upload to enter credentials manually.")

uploaded = {} 
try:
    from google.colab import files
    uploaded = files.upload()
except Exception as e:
    print("Colab file uploader not available or skipped.")

# Set up kaggle credentials
if 'kaggle.json' in uploaded:
    !mkdir -p ~/.kaggle
    with open(os.path.expanduser('~/.kaggle/kaggle.json'), 'wb') as f:
        f.write(uploaded['kaggle.json'])
    !chmod 600 ~/.kaggle/kaggle.json
    print("\nkaggle.json setup complete!")
else:
    print("\nNo kaggle.json uploaded. Sourcing manual credentials:")
    username = os.environ.get('KAGGLE_USERNAME') or input("Kaggle Username: ").strip()
    key = os.environ.get('KAGGLE_KEY') or input("Kaggle API Key: ").strip()
    if username and key:
        os.environ['KAGGLE_USERNAME'] = username
        os.environ['KAGGLE_KEY'] = key
        !mkdir -p ~/.kaggle
        with open(os.path.expanduser('~/.kaggle/kaggle.json'), 'w') as f:
            json.dump({"username": username, "key": key}, f)
        !chmod 600 ~/.kaggle/kaggle.json
        print("Credentials saved to kaggle.json!")
    else:
        raise ValueError("Kaggle credentials are required to continue.")

# Download dataset using Kaggle API
print("\nDownloading snehaanbhawal/resume-dataset from Kaggle...")
!kaggle datasets download -d snehaanbhawal/resume-dataset --unzip

# Locate the CSV file (Resume.csv)
csv_filename = "Resume.csv"
if not os.path.exists(csv_filename):
    # Walk folders to locate CSV
    for root, dirs, files_list in os.walk('.'):
        if 'Resume.csv' in files_list:
            csv_filename = os.path.join(root, 'Resume.csv')
            break

if os.path.exists(csv_filename):
    df = pd.read_csv(csv_filename, encoding="utf-8")
    print(f"\nDataset loaded successfully from {csv_filename}!")
else:
    csv_files = [f for f in os.listdir('.') if f.endswith('.csv')]
    if csv_files:
        csv_filename = csv_files[0]
        df = pd.read_csv(csv_filename, encoding="utf-8")
        print(f"\nDataset loaded successfully from {csv_filename}!")
    else:
        raise FileNotFoundError("Could not locate the unzipped CSV file from Kaggle!")

# Rename the resume column if necessary
if 'Resume' in df.columns:
    df = df.rename(columns={'Resume': 'Resume_str'})

# Drop duplicates to prevent data leakage and inflated F1 scores
original_len = len(df)
df = df.drop_duplicates(subset=['Resume_str'])
df = df.dropna(subset=['Resume_str', 'Category'])

print(f"Total unique resumes: {len(df)} across {df['Category'].nunique()} categories (Removed {original_len - len(df)} duplicate rows to prevent leakage).")
print("Unique Categories:", sorted(df['Category'].unique()))

## Step 3: Define Advanced Text Preprocessing
Define a cleaning function `advanced_clean_resume` to clean and standardize raw resume text before vectorizing. We filter out generic terms to prevent class confusion between software engineering/development and physical roles.

In [ ]:
# Add generic domain stopwords to force the model to focus on actual technical skills
CUSTOM_STOPWORDS = {
    'engineering', 'engineer', 'student', 'professional', 'highly', 'skills',
    'experience', 'work', 'history', 'details', 'academic', 'profile', 'specialize',
    'specializing', 'experienced', 'projects', 'project', 'related', 'responsibility',
    'tutor', 'teaching', 'tutions', 'tutor', 'graphic', 'designer', 'design', 'wordpress',
    'technical', 'like', 'class', 'advanced', 'peer', 'subject', 'idea', 'world', 'home',
    'food', 'school', 'taught', 'grade', 'science'
}
STOP_WORDS = set(stopwords.words('english')).union(CUSTOM_STOPWORDS)
LEMMATIZER = WordNetLemmatizer()

URL_RE = re.compile(r"https?://\S+|www\.\S+")
EMAIL_RE = re.compile(r"\S+@\S+")
NON_ALPHA_RE = re.compile(r"[^a-z\s]")

def advanced_clean_resume(text: str) -> str:
    """Lowercase, strip noise/URLs/emails, drop stopwords, lemmatize."""
    if not isinstance(text, str):
        return ""

    text = text.lower()
    text = URL_RE.sub(" ", text)
    text = EMAIL_RE.sub(" ", text)
    text = NON_ALPHA_RE.sub(" ", text)

    words = text.split()
    words = [
        LEMMATIZER.lemmatize(word)
        for word in words
        if word not in STOP_WORDS and len(word) > 1
    ]
    return " ".join(words)

## Step 4: Feature Extraction, Model Comparison and Training Curves
We clean the dataset, vectorize the texts using TF-IDF, split the dataset, compare different classification algorithms, print the evaluation metrics, and plot the validation curves.

In [ ]:
print("Cleaning resume text... (This will take a few seconds)")
df["Cleaned_Resume"] = df["Resume_str"].apply(advanced_clean_resume)
print("Cleaning Complete!")

X = df["Cleaned_Resume"]
y = df["Category"]
labels = sorted(y.unique())

# TF-IDF Vectorization - tuned max_features and ngram_range for better keyword matching
print("Extracting features using TfidfVectorizer...")
vectorizer = TfidfVectorizer(
    max_features=1500,
    ngram_range=(1, 2),
    min_df=2,
    sublinear_tf=True
)
X_tfidf = vectorizer.fit_transform(X)

# Split dataset (80% Train, 20% Test)
X_train, X_test, y_train, y_test = train_test_split(
    X_tfidf, y, test_size=0.2, random_state=42, stratify=y
)

print("=== SPLIT DETAILS ===")
print(f"Training Samples (80% Data): {X_train.shape[0]}")
print(f"Testing Samples (20% Data): {X_test.shape[0]}")
print(f"Total Extracted Features: {X_train.shape[1]}")

# Candidates for classification
candidates = {
    "Linear SVC": LinearSVC(C=5.0, class_weight="balanced", max_iter=5000, random_state=42),
    "Logistic Regression": LogisticRegression(C=10.0, class_weight="balanced", max_iter=2000, random_state=42),
    "SGD": SGDClassifier(loss="hinge", alpha=0.0001, class_weight="balanced", random_state=42),
    "Multinomial NB": MultinomialNB(alpha=0.01),
}

print("\nComparing models with Cross-Validation (Macro F1)...")
best_name, best_score, best_model = None, -1.0, None
model_names = []
model_scores = []

cv_folds = 5
for name, clf in candidates.items():
    scores = cross_val_score(clf, X_train, y_train, cv=cv_folds, scoring="f1_macro", n_jobs=-1)
    mean_f1 = scores.mean()
    print(f"  {name:<22} macro F1 = {mean_f1:.3f} (+/- {scores.std():.3f})")
    model_names.append(name)
    model_scores.append(mean_f1)
    if mean_f1 > best_score:
        best_name, best_score, best_model = name, mean_f1, clf

print(f"\n--> Selected Best Model: {best_name} with macro F1: {best_score:.3f}")

# Plot 1: Model Comparison Bar Graph
plt.figure(figsize=(8, 5))
plt.bar(model_names, model_scores, color=['#3498db', '#2ecc71', '#9b59b6', '#e67e22'])
plt.ylabel("Macro F1-Score")
plt.title("Model Comparison (5-Fold Cross Validation)")
plt.ylim(0.0, 1.05)
for i, v in enumerate(model_scores):
    plt.text(i, v + 0.02, f"{v:.3f}", ha='center', fontweight='bold')
plt.tight_layout()
plt.show()

# Fit best model on the full training subset
print(f"\nTraining best model ({best_name}) on full training set...")
best_model.fit(X_train, y_train)
train_acc = best_model.score(X_train, y_train)
test_acc = best_model.score(X_test, y_test)
print(f'Training Dataset Accuracy: {train_acc * 100:.2f}%')
print(f'Testing Dataset Accuracy:  {test_acc * 100:.2f}%')

# Plot 2: Learning Curve
print("\nGenerating learning curves...")
train_sizes, train_scores, test_scores = learning_curve(
    best_model, X_train, y_train, cv=cv_folds, scoring="f1_macro", n_jobs=-1,
    train_sizes=np.linspace(0.1, 1.0, 5)
)
train_scores_mean = np.mean(train_scores, axis=1)
train_scores_std = np.std(train_scores, axis=1)
test_scores_mean = np.mean(test_scores, axis=1)
test_scores_std = np.std(test_scores, axis=1)

plt.figure(figsize=(10, 6))
plt.fill_between(train_sizes, train_scores_mean - train_scores_std,
                 train_scores_mean + train_scores_std, alpha=0.1, color="r")
plt.fill_between(train_sizes, test_scores_mean - test_scores_std,
                 test_scores_mean + test_scores_std, alpha=0.1, color="g")
plt.plot(train_sizes, train_scores_mean, 'o-', color="r", label="Training score")
plt.plot(train_sizes, test_scores_mean, 'o-', color="g", label="Cross-validation score")
plt.title(f"Learning Curves (Training Curve) - {best_name}")
plt.xlabel("Training Examples")
plt.ylabel("Macro F1-Score")
plt.grid(True)
plt.legend(loc="best")
plt.show()

# Detailed Classification Report
y_pred = best_model.predict(X_test)
print("\n=== DETAILED CLASSIFICATION REPORT ===")
print(classification_report(y_test, y_pred, target_names=labels))

# Plot 3: Confusion Matrix Heatmap
cm = confusion_matrix(y_test, y_pred, labels=labels)
plt.figure(figsize=(14, 11))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=labels, yticklabels=labels)
plt.title(f'Confusion Matrix - {best_name}', fontsize=16)
plt.xlabel('Predicted Job Profiles', fontsize=12)
plt.ylabel('Actual Job Profiles', fontsize=12)
plt.xticks(rotation=90)
plt.yticks(rotation=0)
plt.tight_layout()
plt.show()

# Fit best model on 100% of data before saving to production
print("\nTraining final production model on full 100% dataset...")
production_model = candidates[best_name]
production_model.fit(X_tfidf, y)

# Save model assets
joblib.dump(production_model, "resume_model.pkl")
joblib.dump(vectorizer, "tfidf_vectorizer.pkl")
print("\nSaved model & vectorizer files successfully!")

## Step 5: Gradio UI Testing Playground
Launch Gradio to interactively test the model. You can paste any resume text and see predictions instantly.

In [ ]:
import gradio as gr

# Confidence calculation helper
def predict_with_confidence(text: str):
    cleaned = advanced_clean_resume(text)
    vector = vectorizer.transform([cleaned])
    prediction = production_model.predict(vector)[0]

    # Calculate probability-like score
    if hasattr(production_model, "predict_proba"):
        probs = production_model.predict_proba(vector)[0]
        confidence = float(probs.max())
    elif hasattr(production_model, "decision_function"):
        scores = production_model.decision_function(vector)[0]
        if hasattr(scores, "__len__") and len(scores) > 1:
            # Applying temperature scaling (T=5) to sharpen confidence logits
            exp_scores = np.exp(5 * (scores - scores.max()))
            probs = exp_scores / exp_scores.sum()
            confidence = float(probs.max())
        else:
            val = float(scores[0]) if hasattr(scores, "__len__") else float(scores)
            confidence = float(1.0 / (1.0 + np.exp(-5 * abs(val))))
    else:
        confidence = None
    return prediction, confidence

def predict_gradio(resume_text):
    if not resume_text.strip():
        return 'Please paste some resume text first.', '0%', ''
    
    category, confidence = predict_with_confidence(resume_text)
    confidence_val = f'{confidence * 100:.1f}%' if confidence is not None else 'N/A'
    return category, confidence_val, best_name

# Create the Gradio interface
demo = gr.Interface(
    fn=predict_gradio,
    inputs=gr.Textbox(lines=10, label='Paste Resume Text', placeholder='Paste resume details, education, skills, and experience here...'),
    outputs=[
        gr.Textbox(label='Predicted Category/Job Role'),
        gr.Textbox(label='Confidence Score'),
        gr.Textbox(label='Classifier Model Used')
    ],
    title='Resume Classification Playground',
    description=f'Interactively test the best-performing model ({best_name}) trained on the 24-category Resume Dataset.',
    theme='soft'
)

# Launch the Gradio Interface (generates an inline UI and public live link)
demo.launch(share=True)